Population Estimate and Confidence Interval

In [ ]:
import math
import pandas as pd

data = {
    "Java": {"Basic": 737, "Intermediate": 408, "Advanced": 102},
    "Python": {"Basic": 218, "Intermediate": 535, "Advanced": 217},
    "C++": {"Basic": 138, "Intermediate": 360, "Advanced": 462},
    "JavaScript": {"Basic": 372, "Intermediate": 448, "Advanced": 182}
}

for lang in data:
    data[lang]["Total"] = sum(data[lang].values())

merged = {"Basic": 0, "Intermediate": 0, "Advanced": 0, "Total": 0}
for lang in data:
    merged["Basic"] += data[lang]["Basic"]
    merged["Intermediate"] += data[lang]["Intermediate"]
    merged["Advanced"] += data[lang]["Advanced"]
    merged["Total"] += data[lang]["Total"]
data["Merged"] = merged

def proportion_ci(count, total):
    p = count / total
    se = math.sqrt(p * (1 - p) / total)
    moe = 1.96 * se * 100 
    return p * 100, moe


results = {}
for lang, vals in data.items():
    total = vals["Total"]
    res = {}
    for cat in ["Basic", "Intermediate", "Advanced"]:
        pct, moe = proportion_ci(vals[cat], total)
        res[cat] = (round(pct, 1), round(moe, 1))
    results[lang] = res


df = pd.DataFrame(results).T
df


,Basic,Intermediate,Advanced
Java,"(59.1, 2.7)","(32.7, 2.6)","(8.2, 1.5)"
Python,"(22.5, 2.6)","(55.2, 3.1)","(22.4, 2.6)"
C++,"(14.4, 2.2)","(37.5, 3.1)","(48.1, 3.2)"
JavaScript,"(37.1, 3.0)","(44.7, 3.1)","(18.2, 2.4)"
Merged,"(35.1, 1.4)","(41.9, 1.5)","(23.0, 1.3)"


Statistical Tests

In [ ]:
import numpy as np
from scipy.stats import shapiro, wilcoxon, ttest_rel
from itertools import combinations

# ===================== MODEL DATA =====================

hasan_f1 = [
    58.4, 56.0, 56.6, 55.9, 57.2,
    67.3, 65.1, 65.8, 64.6, 66.1,
    59.8, 58.2, 58.8, 57.7, 59.1,
    58.7, 56.5, 57.0, 55.9, 57.6,
    60.9, 59.0, 59.3, 58.0, 59.6
]

raida_f1 = [
    58.5, 57.3, 56.8, 55.5, 56.3,
    66.1, 64.2, 64.9, 63.4, 65.3,
    59.4, 57.9, 58.3, 56.8, 58.0,
    60.4, 58.1, 58.6, 57.0, 58.0,
    62.7, 60.9, 61.2, 60.0, 60.7
]

wolora_f1 = [
    63.0, 61.7, 61.0, 60.4, 61.3,
    72.1, 70.0, 70.3, 69.1, 71.0,
    68.9, 66.7, 67.1, 65.9, 67.5,
    67.4, 65.0, 65.8, 64.3, 66.2,
    67.2, 65.0, 65.5, 64.3, 66.0
]

wospear_f1 = [
    64.1, 62.5, 61.8, 60.7, 61.9,
    72.7, 71.0, 71.4, 70.3, 71.9,
    70.8, 69.0, 69.5, 68.0, 69.9,
    69.0, 66.7, 67.3, 65.7, 67.7,
    68.5, 66.3, 66.7, 65.0, 67.0
]

wosmote_f1 = [
    63.7, 62.2, 61.5, 60.5, 61.6,
    72.5, 70.8, 71.2, 70.1, 71.7,
    70.4, 68.6, 69.1, 67.6, 67.3,
    68.7, 66.5, 67.0, 65.5, 67.3,
    68.2, 66.0, 66.5, 64.9, 66.7
]

stackranker_f1 = [
    64.6, 63.0, 62.1, 61.5, 62.5,
    74.4, 72.6, 73.0, 71.9, 73.4,
    71.5, 69.6, 70.0, 68.7, 70.6,
    70.0, 68.2, 68.9, 67.3, 69.1,
    69.3, 67.0, 67.5, 65.9, 67.8
]

hasan_acc = [
    58.4, 56.2, 56.8, 55.7, 57.3,
    67.9, 65.5, 66.0, 64.9, 66.7,
    60.4, 58.9, 59.2, 58.3, 59.6,
    59.0, 56.7, 57.2, 56.1, 58.0,
    61.5, 59.7, 60.0, 58.6, 60.3
]

raida_acc = [
    62.8, 61.1, 60.5, 59.8, 60.2,
    68.9, 66.4, 67.0, 65.9, 67.3,
    66.5, 64.7, 65.2, 63.9, 65.0,
    62.5, 60.1, 60.8, 59.3, 60.2,
    63.2, 61.4, 61.8, 60.6, 61.1
]

wolora_acc = [
    62.5, 61.4, 60.8, 60.1, 61.0,
    72.5, 70.3, 70.8, 69.6, 71.4,
    69.5, 67.4, 67.8, 66.5, 68.2,
    67.8, 65.5, 66.2, 64.8, 66.6,
    67.3, 65.3, 65.8, 64.5, 66.2
]

wospear_acc = [
    64.2, 62.7, 61.9, 60.9, 62.1,
    73.2, 71.5, 71.9, 70.7, 72.4,
    71.0, 69.2, 69.7, 68.3, 70.1,
    69.3, 67.1, 67.7, 66.1, 68.1,
    68.6, 66.5, 67.0, 65.3, 67.2
]

wosmote_acc = [
    63.9, 62.4, 61.6, 60.6, 61.8,
    72.3, 71.2, 71.6, 70.5, 72.1,
    70.6, 68.8, 69.3, 67.9, 69.7,
    68.9, 66.8, 67.4, 65.8, 67.7,
    68.3, 66.2, 66.7, 65.1, 66.9
]

stackranker_acc = [
    64.7, 63.1, 62.4, 61.8, 62.7,
    74.6, 72.8, 73.2, 72.1, 73.6,
    71.6, 69.8, 70.2, 68.9, 70.9,
    70.9, 68.8, 69.4, 67.9, 69.7,
    69.3, 67.2, 67.6, 66.1, 68.0
]

# ===================== Helper: Cohen's d =====================

def cohens_d(x, y):
    diff = np.array(x) - np.array(y)
    return np.mean(diff) / np.std(diff, ddof=1)

# ===================== NORMALITY TEST =====================

models_f1 = {
    "Hasan": hasan_f1,
    "Raida": raida_f1,
    "WoLoRA": wolora_f1,
    "WoSMOTE": wosmote_f1,
    "WoSPEAR": wospear_f1,
    "StackRanker": stackranker_f1
}

models_acc = {
    "Hasan": hasan_acc,
    "Raida": raida_acc,
    "WoLoRA": wolora_acc,
    "WoSMOTE": wosmote_acc,
    "WoSPEAR": wospear_acc,
    "StackRanker": stackranker_acc
}

alpha = 0.05

print("=== Shapiro–Wilk Test for Normality ===")
for metric_name, models in [("F1", models_f1), ("Accuracy", models_acc)]:
    print(f"\n-- {metric_name} --")
    for name, scores in models.items():
        stat, p = shapiro(scores)
        normality = "Normal" if p > alpha else "Non-normal"
        print(f"{name:<12} → W={stat:.4f}, p={p:.4e}, {normality}")

# ===================== PAIRWISE TESTS (Wilcoxon, t-test, Cohen's d) =====================

def pairwise_tests(models, metric_name):
    names = list(models.keys())
    print(f"\n=== Pairwise Statistical Tests ({metric_name}) ===")
    for (m1, m2) in combinations(names, 2):
        x, y = models[m1], models[m2]
        # Wilcoxon
        w_stat, w_p = wilcoxon(x, y)
        # Paired t-test
        t_stat, t_p = ttest_rel(x, y)
        # Cohen's d
        d = cohens_d(x, y)

        print(f"\n{m1} vs {m2}")
        print(f"  Wilcoxon → stat={w_stat:.4f}, p={w_p:.4e} {'(sig)' if w_p < alpha else ''}")
        print(f"  t-test   → stat={t_stat:.4f}, p={t_p:.4e} {'(sig)' if t_p < alpha else ''}")
        print(f"  Cohen's d → {d:.4f} ({'small' if abs(d)<0.5 else 'medium' if abs(d)<0.8 else 'large'} effect)")

pairwise_tests(models_f1, "F1")
pairwise_tests(models_acc, "Accuracy")

print("\nAnalysis complete ✅")


=== Shapiro–Wilk Test for Normality ===

-- F1 --
Hasan        → W=0.8343, p=8.9816e-04, Non-normal
Raida        → W=0.9213, p=5.4888e-02, Normal
WoLoRA       → W=0.9745, p=7.6005e-01, Normal
WoSMOTE      → W=0.9623, p=4.6237e-01, Normal
WoSPEAR      → W=0.9593, p=3.9980e-01, Normal
StackRanker  → W=0.9577, p=3.7014e-01, Normal

-- Accuracy --
Hasan        → W=0.8692, p=4.1626e-03, Non-normal
Raida        → W=0.9206, p=5.2833e-02, Normal
WoLoRA       → W=0.9657, p=5.3915e-01, Normal
WoSMOTE      → W=0.9526, p=2.8724e-01, Normal
WoSPEAR      → W=0.9615, p=4.4426e-01, Normal
StackRanker  → W=0.9522, p=2.8115e-01, Normal

=== Pairwise Statistical Tests (F1) ===

Hasan vs Raida
  Wilcoxon → stat=117.0000, p=2.3036e-01 
  t-test   → stat=-1.2216, p=2.3371e-01 
  Cohen's d → -0.2443 (small effect)

Hasan vs WoLoRA
  Wilcoxon → stat=0.0000, p=5.9605e-08 (sig)
  t-test   → stat=-18.2128, p=1.4816e-15 (sig)
  Cohen's d → -3.6426 (large effect)

Hasan vs WoSMOTE
  Wilcoxon → stat=0.0000, p=5.960

In [ ]:
import numpy as np
from scipy.stats import friedmanchisquare


hasan_f1 = [
    # Python
    58.4, 56.0, 56.6, 55.9, 57.2,
    # Java
    67.3, 65.1, 65.8, 64.6, 66.1,
    # JavaScript
    59.8, 58.2, 58.8, 57.7, 59.1,
    # C++
    58.7, 56.5, 57.0, 55.9, 57.6,
    # Merged
    60.9, 59.0, 59.3, 58.0, 59.6
]

raida_f1 = [
    58.5, 57.3, 56.8, 55.5, 56.3,
    66.1, 64.2, 64.9, 63.4, 65.3,
    59.4, 57.9, 58.3, 56.8, 58.0,
    60.4, 58.1, 58.6, 57.0, 58.0,
    62.7, 60.9, 61.2, 60.0, 60.7
]

wolora_f1 = [
    63.0, 61.7, 61.0, 60.4, 61.3,
    72.1, 70.0, 70.3, 69.1, 71.0,
    68.9, 66.7, 67.1, 65.9, 67.5,
    67.4, 65.0, 65.8, 64.3, 66.2,
    67.2, 65.0, 65.5, 64.3, 66.0
]

wospear_f1 = [
    64.1, 62.5, 61.8, 60.7, 61.9,
    72.7, 71.0, 71.4, 70.3, 71.9,
    70.8, 69.0, 69.5, 68.0, 69.9,
    69.0, 66.7, 67.3, 65.7, 67.7,
    68.5, 66.3, 66.7, 65.0, 67.0
]

wosmote_f1 = [
    63.7, 62.2, 61.5, 60.5, 61.6,
    72.5, 70.8, 71.2, 70.1, 71.7,
    70.4, 68.6, 69.1, 67.6, 67.3,
    68.7, 66.5, 67.0, 65.5, 67.3,
    68.2, 66.0, 66.5, 64.9, 66.7
]

stackranker_f1 = [
    64.6, 63.0, 62.1, 61.5, 62.5,
    74.4, 72.6, 73.0, 71.9, 73.4,
    71.5, 69.6, 70.0, 68.7, 70.6,
    70.0, 68.2, 68.9, 67.3, 69.1,
    69.3, 67.0, 67.5, 65.9, 67.8
]


# ================= ACCURACY SCORES =================

hasan_acc = [
    58.4, 56.2, 56.8, 55.7, 57.3,
    67.9, 65.5, 66.0, 64.9, 66.7,
    60.4, 58.9, 59.2, 58.3, 59.6,
    59.0, 56.7, 57.2, 56.1, 58.0,
    61.5, 59.7, 60.0, 58.6, 60.3
]

raida_acc = [
    62.8, 61.1, 60.5, 59.8, 60.2,
    68.9, 66.4, 67.0, 65.9, 67.3,
    66.5, 64.7, 65.2, 63.9, 65.0,
    62.5, 60.1, 60.8, 59.3, 60.2,
    63.2, 61.4, 61.8, 60.6, 61.1
]

wolora_acc = [
    62.5, 61.4, 60.8, 60.1, 61.0,
    72.5, 70.3, 70.8, 69.6, 71.4,
    69.5, 67.4, 67.8, 66.5, 68.2,
    67.8, 65.5, 66.2, 64.8, 66.6,
    67.3, 65.3, 65.8, 64.5, 66.2
]

wospear_acc = [
    64.2, 62.7, 61.9, 60.9, 62.1,
    73.2, 71.5, 71.9, 70.7, 72.4,
    71.0, 69.2, 69.7, 68.3, 70.1,
    69.3, 67.1, 67.7, 66.1, 68.1,
    68.6, 66.5, 67.0, 65.3, 67.2
]

wosmote_acc = [
    63.9, 62.4, 61.6, 60.6, 61.8,
    72.3, 71.2, 71.6, 70.5, 72.1,
    70.6, 68.8, 69.3, 67.9, 69.7,
    68.9, 66.8, 67.4, 65.8, 67.7,
    68.3, 66.2, 66.7, 65.1, 66.9
]

stackranker_acc = [
    64.7, 63.1, 62.4, 61.8, 62.7,
    74.6, 72.8, 73.2, 72.1, 73.6,
    71.6, 69.8, 70.2, 68.9, 70.9,
    70.9, 68.8, 69.4, 67.9, 69.7,
    69.3, 67.2, 67.6, 66.1, 68.0
]



# Friedman Test 
stat_f1, p_f1 = friedmanchisquare(
    hasan_f1, raida_f1, wolora_f1, wosmote_f1, wospear_f1, stackranker_f1
)
stat_acc, p_acc = friedmanchisquare(
    hasan_acc, raida_acc, wolora_acc, wosmote_acc, wospear_acc, stackranker_acc
)

# Results
print("=== Friedman Test Across Models ===")
print(f"F1  → χ² = {stat_f1:.4f}, p = {p_f1:.4e}")
print(f"Acc → χ² = {stat_acc:.4f}, p = {p_acc:.4e}")

alpha = 0.05
sig_f1 = "Yes" if p_f1 < alpha else "No"
sig_acc = "Yes" if p_acc < alpha else "No"

print("\nInterpretation:")
print(f"F1  → Significant difference among models? {sig_f1}")
print(f"Acc → Significant difference among models? {sig_acc}")


=== Friedman Test Across Models ===
F1  → χ² = 120.8857, p = 2.0376e-24
Acc → χ² = 123.8800, p = 4.7271e-25

Interpretation:
F1  → Significant difference among models? Yes
Acc → Significant difference among models? Yes
